# Week 10 — Block 2: Guided Demo (Maps II)

**DATS 6401 · Visualization of Complex Data**

~35 min. Last week's choropleth → interactive → embedded in a Streamlit dashboard.

1. folium: the choropleth, now zoomable, with tooltips (~12 min)
2. Save/inspect the HTML — what a web map IS (~5 min)
3. The dashboard: `dashboard_starter.py`, coordinated filter (~18 min)

*(Cells 1–2 run offline; the map tiles need internet to DISPLAY, but the HTML generates regardless.)*

In [ ]:
import geopandas as gpd
import pandas as pd
import json

gdf = gpd.read_file("../week9/data/counties.geojson").set_crs(epsg=4326, allow_override=True)
stats = pd.read_csv("../week9/data/county_stats.csv")
joined = gdf.merge(stats, on="county_id")
joined["rate"] = joined["events"] / joined["population"] * 1000
print(len(joined), "counties ready")

## Part 1 — The interactive choropleth

In [ ]:
import folium

minx, miny, maxx, maxy = joined.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]   # bounds midpoint: no CRS warning, no centroid math
m = folium.Map(location=center, zoom_start=8, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=json.loads(joined.to_json()),
    data=joined,
    columns=["county_id", "rate"],
    key_on="feature.properties.county_id",     # <- the join between layers
    fill_color="OrRd",
    legend_name="events per 1,000",
).add_to(m)

# tooltips: hover for details-on-demand
folium.GeoJson(
    json.loads(joined.to_json()),
    style_function=lambda f: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(fields=["name", "rate"], aliases=["County", "Rate/1k"]),
).add_to(m)

m.save("week10_map.html")
print("wrote week10_map.html — open it in a browser")
m

**Narrate `key_on`:** folium joins YOUR dataframe to the GeoJSON by this property path — the geographic equivalent of a merge key, and the #1 source of blank maps when it's wrong.

## Part 2 — What did we just make?

Open `week10_map.html` in a text editor (briefly): it's self-contained HTML+JS — Leaflet, your GeoJSON inlined, the interaction wiring. **That file is the deliverable** a web map produces; Streamlit will simply host it.

## Part 3 — The dashboard

Switch to `dashboard_starter.py` (this folder) and run:

```bash
streamlit run dashboard_starter.py
```

Build order to narrate (matching the deck's coordination pattern):

1. The **metric selectbox** — ONE control
2. The folium map, embedded via `st_folium`, driven by that metric
3. The **ranking bar chart** — same metric, position channel (maps can't rank!)
4. `out = st_folium(...)` → `out["last_object_clicked"]` → the details panel

Kill move for the room: click a county on the map and watch the panel update — that's Shneiderman's *details on demand*, in 6 lines of the starter.